# Notebook 10: Comprehensive Signal Quality Index (SQI) Analysis

## Objectives
- Compute all SQI metrics defined in the pulse morphology plan
- Visualize SQI distributions across trials
- Train a good/bad signal quality classifier
- Apply SQIs to rPPG and BVP signals for quality assessment


In [ ]:
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')
from src.config import ALIGNED_DIR, FEATURES_DIR, FIGURES_DIR, ARCHIVE_DIR


In [ ]:
# Load aligned data and feature table
aligned_files = sorted(Path(ALIGNED_DIR).glob('*_aligned_1hz.csv'))
print(f'Found {len(aligned_files)} aligned trials')
features = pd.read_csv(FEATURES_DIR / 'trial_features.csv')
features.head(3)


In [ ]:
from src.signals.comprehensive_sqi import (
    compute_all_sqis, classify_signal_quality, n_sqi, s_sqi, k_sqi,
    e_sqi, z_sqi, r_sqi, m_sqi, periodicity, peak_stability, pulse_consistency
)


In [ ]:
# Compute SQI metrics for each trial's BVP signal
from src.data.empatica_loader import load_trial_signals
from src.data.archive_scanner import scan_archive

records = scan_archive(ARCHIVE_DIR)
sqi_rows = []
for rec in records:
    try:
        signals, meta = load_trial_signals(rec.files)
        if 'BVP' not in signals:
            continue
        bvp = signals['BVP']['bvp'].to_numpy()
        sr = meta['BVP']['sample_rate']
        sqi_dict = compute_all_sqis(bvp, sr, prefix='bvp')
        sqi_dict['subject_id'] = rec.subject_id
        sqi_dict['trial_id'] = rec.trial_id
        sqi_dict['quality_class'] = classify_signal_quality(sqi_dict)
        sqi_rows.append(sqi_dict)
    except Exception as e:
        print(f'Error processing {rec.subject_id}/{rec.trial_id}: {e}')

sqi_df = pd.DataFrame(sqi_rows)
print(f'Computed SQIs for {len(sqi_df)} trials')
sqi_df.head(3)


In [ ]:
# SQI Distribution Visualizations
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
sqi_metrics = ['bvp_n_sqi', 'bvp_s_sqi', 'bvp_k_sqi', 'bvp_e_sqi',
               'bvp_z_sqi', 'bvp_r_sqi', 'bvp_periodicity',
               'bvp_peak_stability', 'bvp_pulse_consistency']
titles = ['N_SQI (SNR)', 'S_SQI (Skewness)', 'K_SQI (Kurtosis)',
          'E_SQI (Entropy)', 'Z_SQI (Zero-Crossing)', 'R_SQI (Spectral Ratio)',
          'Periodicity', 'Peak Stability', 'Pulse Consistency']
for ax, metric, title in zip(axes.flat, sqi_metrics, titles):
    if metric in sqi_df:
        vals = sqi_df[metric].dropna()
        ax.hist(vals, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
        ax.set_title(f'{title}\n(n={len(vals)}, mean={vals.mean():.2f})')
        ax.set_xlabel('')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sqi_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Quality Class Distribution
if 'quality_class' in sqi_df:
    counts = sqi_df['quality_class'].value_counts()
    fig, ax = plt.subplots(figsize=(8, 5))
    colors = {'Excellent': 'green', 'Acceptable': 'orange', 'Unfit': 'red'}
    bars = ax.bar(counts.index, counts.values,
                  color=[colors.get(c, 'gray') for c in counts.index])
    ax.set_title('Signal Quality Distribution Across Trials', fontsize=14)
    ax.set_ylabel('Number of Trials')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                str(val), ha='center', fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'sqi_quality_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(counts)


In [ ]:
# Train Signal Quality Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, RocCurveDisplay

# Prepare features for classification
feature_cols = ['bvp_n_sqi', 'bvp_s_sqi', 'bvp_k_sqi', 'bvp_e_sqi',
               'bvp_z_sqi', 'bvp_r_sqi', 'bvp_periodicity']
X = sqi_df[feature_cols].fillna(0)

# Create binary target: Good (Excellent+Acceptable) vs Bad (Unfit)
y = (sqi_df['quality_class'] != 'Unfit').astype(int)
print(f'Good signals: {y.sum()}, Bad signals: {(1-y).sum()}')

# Train RF classifier with cross-validation
clf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(clf, X, y, cv=min(5, len(X)), scoring='accuracy')
print(f'Cross-val accuracy: {scores.mean():.3f} +/- {scores.std():.3f}')

clf.fit(X, y)

# Feature importance
importances = pd.DataFrame({'feature': feature_cols, 'importance': clf.feature_importances_})
importances = importances.sort_values('importance', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(importances['feature'], importances['importance'], color='teal')
ax.set_title('SQI Feature Importance for Signal Quality Classification', fontsize=14)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sqi_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print(importances)


In [ ]:
# Apply SQIs to rPPG signals from video processing
from src.video.rppg import extract_video_rgb_traces

video_records = [r for r in records if r.video_path is not None]
if video_records:
    rec = video_records[0]
    print(f'Processing video: {rec.video_path}')
    traces = extract_video_rgb_traces(rec.video_path, max_frames=300)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(traces['time_seconds'], traces['rppg_green'], color='green', lw=0.8)
    axes[0].set_title('rPPG - Green Channel'), axes[0].set_ylabel('Amplitude')
    axes[1].plot(traces['time_seconds'], traces['rppg_chrom'], color='blue', lw=0.8)
    axes[1].set_title('rPPG - CHROM'), axes[1].set_ylabel('Amplitude')
    axes[2].plot(traces['time_seconds'], traces['rppg_pos'], color='red', lw=0.8)
    axes[2].set_title('rPPG - POS'), axes[2].set_xlabel('Time (s)'), axes[2].set_ylabel('Amplitude')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'rppg_signals_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Compute SQIs for each rPPG method
    fps = 30.0
    for method, col in [('Green', 'rppg_green'), ('CHROM', 'rppg_chrom'), ('POS', 'rppg_pos')]:
        sig = traces[col].dropna().to_numpy()
        sqis = compute_all_sqis(sig, fps, prefix='rppg')
        qclass = classify_signal_quality(sqis)
        snr_v = sqis.get("rppg_n_sqi", 0)
        skew_v = sqis.get("rppg_s_sqi", 0)
        kurt_v = sqis.get("rppg_k_sqi", 0)
        per_v = sqis.get("rppg_periodicity", 0)
        print(f'{method}: SNR={snr_v:.1f}dB, Skew={skew_v:.2f}, Kurt={kurt_v:.2f}, Period={per_v:.2f}, Quality={qclass}')

print('SQI Analysis complete.')
